# FlexPose 虚拟筛选教程

**论文**: Dong et al., *FlexPose: A Framework for End-to-End Flexible Molecular Docking*, NeurIPS 2023 (Submitted)

**核心创新**: FlexPose 实现了真正的端到端**柔性对接**（flexible docking）。与传统半柔性对接方法（如 DiffDock、KarmaDock）只预测配体位姿不同，FlexPose 同时对蛋白质口袋侧链和配体施加噪声，通过双向消息传递层联合更新两者的特征与坐标，最终**同时预测蛋白质侧链坐标和配体坐标**。

**关键教学要点**:
1. 同时对蛋白质口袋侧链原子和配体原子施加噪声（模拟未知构象）
2. 通过双向消息传递层联合更新蛋白质和配体的特征与坐标
3. 最终同时预测蛋白质侧链坐标和配体坐标（联合坐标预测）

**学习目标**:
- 理解柔性对接与半柔性对接的关键区别
- 掌握双向消息传递的原理：蛋白-配体交互如何同时更新两侧坐标
- 理解加噪-去噪训练范式在坐标预测中的应用
- 学会用 RMSD 衡量对接质量（配体 RMSD + 蛋白侧链 RMSD）

In [ ]:
# ============================================================
# 项目根目录定位、路径设置与依赖导入
# ============================================================
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from rdkit import Chem
from rdkit import RDLogger
import matplotlib.pyplot as plt

%matplotlib inline

# 抑制 RDKit 的冗余警告信息
RDLogger.logger().setLevel(RDLogger.ERROR)
warnings.filterwarnings('ignore')


def find_project_root(marker='demo_data'):
    """向上查找包含 demo_data/ 的项目根目录"""
    here = Path('.').resolve()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f'无法找到包含 {marker}/ 的项目根目录')


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'demo_data'
CORESET_FILE = DATA_DIR / 'CoreSet.dat'
COMPLEX_DIR = DATA_DIR / 'coreset'

print(f'项目根目录: {PROJECT_ROOT}')
print(f'数据目录:   {DATA_DIR}')

# 版本信息
version_info = pd.DataFrame({
    '库': ['torch', 'rdkit', 'numpy', 'pandas'],
    '版本': [
        torch.__version__,
        Chem.rdBase.rdkitVersion,
        np.__version__,
        pd.__version__,
    ]
})
display(version_info)

## 1. 超参数设置

FlexPose 的超参数分为两大类：
- **噪声参数**: `SIGMA_PROT` 和 `SIGMA_LIG` 控制训练时添加到蛋白质侧链和配体坐标的高斯噪声标准差。蛋白侧链噪声较小（1.0 A），因为侧链构象变化有限；配体噪声较大（5.0 A），模拟初始位姿完全未知。
- **模型参数**: 隐层维度、交互层数、距离阈值等控制模型的表达能力和计算开销。

In [ ]:
# ============================================================
# 超参数定义
# ============================================================
DISTANCE_CUTOFF = 6.0       # 交互边的距离阈值 (Angstrom)
ATOM_FEAT_DIM = 10          # 简化原子特征维度
HIDDEN_DIM = 128            # 隐层维度
N_LAYERS = 6                # 交互层数
N_EPOCHS = 200              # 训练轮数
LR = 1e-3                   # 学习率
BATCH_SIZE = 1              # 变长图，逐样本处理
SEED = 42
SIGMA_PROT = 1.0            # 蛋白质坐标噪声标准差 (Angstrom)
SIGMA_LIG = 5.0             # 配体坐标噪声标准差 (Angstrom)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)

# 展示超参数
hp_df = pd.DataFrame({
    '超参数': ['DISTANCE_CUTOFF', 'ATOM_FEAT_DIM', 'HIDDEN_DIM', 'N_LAYERS',
              'N_EPOCHS', 'LR', 'BATCH_SIZE', 'SIGMA_PROT', 'SIGMA_LIG', 'DEVICE'],
    '值': [DISTANCE_CUTOFF, ATOM_FEAT_DIM, HIDDEN_DIM, N_LAYERS,
           N_EPOCHS, LR, BATCH_SIZE, SIGMA_PROT, SIGMA_LIG, str(DEVICE)],
    '说明': ['交互边距离阈值 (A)', '原子特征维度', '隐层维度', '双向交互层数',
            '训练轮数', '学习率', '每批样本数', '蛋白坐标噪声 sigma (A)',
            '配体坐标噪声 sigma (A)', '计算设备']
})
display(hp_df)

## 2. 数据加载与特征提取

数据来自 PDBbind CASF-2016 核心集（285 个蛋白-配体复合物）。对每个复合物，我们提取：
- **蛋白口袋**: 从 `{pdbid}_pocket.pdb` 读取，去氢后提取重原子坐标和特征
- **配体**: 优先从 `{pdbid}_ligand.mol2` 读取，备选 `.sdf` 格式
- **原子特征**: 10 维向量 = 9 维元素 one-hot（C, N, O, S, F, P, Cl, Br, other）+ 1 维芳香性标志

此外，定义了几个辅助函数：
- `random_rotation_matrix()`: 数据增强用的随机旋转
- `add_noise_to_coords()`: 给坐标添加高斯噪声
- `compute_rmsd()`: 计算预测坐标与真实坐标的 RMSD

In [ ]:
# ============================================================
# 特征提取函数
# ============================================================

# ---- 元素符号 -> one-hot 映射 (9类 + 1 芳香性 = 10维) ----
ELEMENT_LIST = ['C', 'N', 'O', 'S', 'F', 'P', 'Cl', 'Br']  # 8 种常见元素 + 1 other


def parse_coreset(path):
    """解析 CoreSet.dat，返回 pdbid 列表。跳过以 '#' 开头的注释行。"""
    pdbids = []
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            pdbids.append(parts[0])
    print(f"[INFO] 从 CoreSet.dat 读取到 {len(pdbids)} 个复合物")
    return pdbids


def atom_features(atom):
    """
    构建 10 维原子特征向量：
      - 前 9 维: 元素类型 one-hot (C, N, O, S, F, P, Cl, Br, other)
      - 第 10 维: 是否为芳香性原子
    """
    feat = np.zeros(ATOM_FEAT_DIM, dtype=np.float32)
    symbol = atom.GetSymbol()
    if symbol in ELEMENT_LIST:
        feat[ELEMENT_LIST.index(symbol)] = 1.0
    else:
        feat[8] = 1.0       # other 类别
    feat[9] = float(atom.GetIsAromatic())
    return feat


def random_rotation_matrix():
    """
    生成随机 3D 旋转矩阵（用于数据增强）。
    基于 QR 分解生成均匀分布的旋转矩阵。
    """
    H = np.random.randn(3, 3)
    Q, R = np.linalg.qr(H)
    # 确保是正交旋转矩阵 (det = +1)
    Q = Q @ np.diag(np.sign(np.diag(R)))
    if np.linalg.det(Q) < 0:
        Q[:, 0] *= -1
    return Q.astype(np.float32)


def add_noise_to_coords(coords, sigma):
    """
    给坐标添加高斯噪声。
    coords: (N, 3) numpy 数组
    sigma: 噪声标准差 (Angstrom)
    返回: 加噪后的坐标 (N, 3)
    """
    noise = np.random.randn(*coords.shape).astype(np.float32) * sigma
    return coords + noise


def compute_rmsd(pred, true):
    """
    计算 RMSD (Root Mean Square Deviation)。
    pred, true: (N, 3) torch 张量
    """
    diff = pred - true
    return torch.sqrt((diff ** 2).sum(dim=-1).mean())


def load_mol(path, fmt):
    """
    用 RDKit 加载分子文件。
    fmt: 'mol2', 'sdf', 'pdb'
    先尝试正常加载，再尝试 sanitize=False 后手动 sanitize。
    """
    mol = None
    if fmt == 'mol2':
        mol = Chem.MolFromMol2File(path, sanitize=True)
        if mol is None:
            mol = Chem.MolFromMol2File(path, sanitize=False)
            if mol is not None:
                try:
                    Chem.SanitizeMol(mol)
                except Exception:
                    pass
    elif fmt == 'sdf':
        supplier = Chem.SDMolSupplier(path, sanitize=True)
        for m in supplier:
            if m is not None:
                mol = m
                break
        if mol is None:
            supplier = Chem.SDMolSupplier(path, sanitize=False)
            for m in supplier:
                if m is not None:
                    mol = m
                    try:
                        Chem.SanitizeMol(mol)
                    except Exception:
                        pass
                    break
    elif fmt == 'pdb':
        mol = Chem.MolFromPDBFile(path, removeHs=False, sanitize=True)
        if mol is None:
            mol = Chem.MolFromPDBFile(path, removeHs=False, sanitize=False)
            if mol is not None:
                try:
                    Chem.SanitizeMol(mol)
                except Exception:
                    pass
    return mol


def build_flexpose_data(pdbid):
    """
    为单个蛋白-配体复合物构建柔性对接数据。

    返回:
      prot_feats  : (N_p, 10)  蛋白口袋原子特征
      prot_coords : (N_p, 3)   蛋白口袋原子真实坐标
      lig_feats   : (N_l, 10)  配体原子特征
      lig_coords  : (N_l, 3)   配体原子真实坐标
    如果解析失败则返回 None。
    """
    pocket_path = os.path.join(str(COMPLEX_DIR), pdbid, f"{pdbid}_pocket.pdb")
    ligand_mol2 = os.path.join(str(COMPLEX_DIR), pdbid, f"{pdbid}_ligand.mol2")
    ligand_sdf = os.path.join(str(COMPLEX_DIR), pdbid, f"{pdbid}_ligand.sdf")

    # ---- 1. 加载蛋白口袋 ----
    prot_mol = Chem.MolFromPDBFile(pocket_path, removeHs=True, sanitize=False)
    if prot_mol is None:
        return None
    try:
        Chem.SanitizeMol(prot_mol)
    except Exception:
        pass

    # ---- 2. 加载配体 (mol2 优先, sdf 备选) ----
    lig_mol = Chem.MolFromMol2File(ligand_mol2, sanitize=False)
    if lig_mol is None and os.path.exists(ligand_sdf):
        lig_mol = load_mol(ligand_sdf, 'sdf')
    if lig_mol is None:
        return None
    try:
        Chem.SanitizeMol(lig_mol)
    except Exception:
        pass

    # ---- 3. 去氢 ----
    try:
        prot_mol = Chem.RemoveHs(prot_mol)
    except Exception:
        pass
    try:
        lig_mol = Chem.RemoveHs(lig_mol)
    except Exception:
        pass

    # ---- 4. 提取 3D 坐标与原子特征 ----
    prot_conf = prot_mol.GetConformer()
    lig_conf = lig_mol.GetConformer()

    prot_coords = np.array([prot_conf.GetAtomPosition(i)
                            for i in range(prot_mol.GetNumAtoms())], dtype=np.float32)
    lig_coords = np.array([lig_conf.GetAtomPosition(i)
                           for i in range(lig_mol.GetNumAtoms())], dtype=np.float32)

    prot_feats = np.array([atom_features(a) for a in prot_mol.GetAtoms()], dtype=np.float32)
    lig_feats = np.array([atom_features(a) for a in lig_mol.GetAtoms()], dtype=np.float32)

    if prot_feats.shape[0] == 0 or lig_feats.shape[0] == 0:
        return None

    return prot_feats, prot_coords, lig_feats, lig_coords

In [ ]:
# ============================================================
# 加载数据并展示一个样本
# ============================================================
print("正在加载蛋白-配体复合物数据...")
pdbids = parse_coreset(str(CORESET_FILE))

all_data = []
failed = 0
for pdbid in sorted(pdbids):
    result = build_flexpose_data(pdbid)
    if result is None:
        failed += 1
        continue
    all_data.append(result)

print(f"成功加载 {len(all_data)} 个复合物, 失败 {failed} 个")

# 展示第一个样本的基本信息
sample = all_data[0]
sample_info = pd.DataFrame({
    '数据': ['蛋白口袋原子特征', '蛋白口袋原子坐标', '配体原子特征', '配体原子坐标'],
    '形状': [str(sample[0].shape), str(sample[1].shape),
            str(sample[2].shape), str(sample[3].shape)],
    '说明': ['(N_p, 10) 元素one-hot + 芳香性', '(N_p, 3) 真实3D坐标',
            '(N_l, 10) 元素one-hot + 芳香性', '(N_l, 3) 真实3D坐标']
})
print("\n第一个样本的数据结构:")
display(sample_info)

## 3. 数据集与数据加载器

FlexPose 的数据集设计有一个关键特点：在 `__getitem__` 时**动态添加噪声**，每次取样本时都会生成不同的噪声，相当于隐式的数据增强。

- **蛋白侧链**: 添加小噪声（sigma=1.0A），模拟侧链的柔性摆动
- **配体**: 添加大噪声（sigma=5.0A），模拟初始位姿完全未知的情况

模型的训练目标就是从加噪坐标恢复出真实坐标（去噪）。

In [ ]:
# ============================================================
# Dataset 与 DataLoader
# ============================================================

class FlexPoseDataset(Dataset):
    """
    柔性对接数据集。
    每个样本包含蛋白口袋和配体的原子特征与真实坐标。
    在 __getitem__ 时动态添加噪声，模拟未知构象作为模型输入。
    """
    def __init__(self, data_list):
        # data_list: list of (prot_feats, prot_coords, lig_feats, lig_coords)
        self.data = data_list

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        prot_f, prot_c, lig_f, lig_c = self.data[idx]

        # ---- 对真实坐标添加噪声，生成模型输入坐标 ----
        # 蛋白侧链坐标加小噪声 (sigma=1.0A)，模拟侧链柔性
        prot_c_noised = add_noise_to_coords(prot_c, SIGMA_PROT)
        # 配体坐标加大噪声 (sigma=5.0A)，模拟初始位姿未知
        lig_c_noised = add_noise_to_coords(lig_c, SIGMA_LIG)

        return (torch.FloatTensor(prot_f),          # (N_p, 10) 蛋白特征
                torch.FloatTensor(prot_c),          # (N_p, 3)  蛋白真实坐标
                torch.FloatTensor(prot_c_noised),   # (N_p, 3)  蛋白加噪坐标
                torch.FloatTensor(lig_f),           # (N_l, 10) 配体特征
                torch.FloatTensor(lig_c),           # (N_l, 3)  配体真实坐标
                torch.FloatTensor(lig_c_noised))    # (N_l, 3)  配体加噪坐标


# ---- 随机 80/20 划分训练/测试集 ----
indices = np.random.permutation(len(all_data))
split = int(0.8 * len(all_data))
train_idx, test_idx = indices[:split], indices[split:]

train_data = [all_data[i] for i in train_idx]
test_data = [all_data[i] for i in test_idx]

print(f"训练集: {len(train_data)} 个复合物, 测试集: {len(test_data)} 个复合物")

train_loader = DataLoader(FlexPoseDataset(train_data), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(FlexPoseDataset(test_data), batch_size=BATCH_SIZE, shuffle=False)

# 展示一个 batch 的数据形状
sample_batch = next(iter(train_loader))
batch_shapes = pd.DataFrame({
    '张量': ['prot_feats', 'prot_coords_true', 'prot_coords_noised',
            'lig_feats', 'lig_coords_true', 'lig_coords_noised'],
    '形状': [str(tuple(t.shape)) for t in sample_batch],
    '说明': ['蛋白原子特征', '蛋白真实坐标', '蛋白加噪坐标',
            '配体原子特征', '配体真实坐标', '配体加噪坐标']
})
print("\n单个 batch 的张量形状:")
display(batch_shapes)

## 4. 模型架构

ToyFlexPose 模型由以下组件构成：

1. **原子特征嵌入层**: 蛋白和配体使用**独立的**嵌入层（`prot_embed`, `lig_embed`），将 10 维原子特征映射到 128 维隐空间

2. **双向交互层** (`BidirectionalInteractionLayer`): 这是 FlexPose 的核心创新
   - 计算所有原子对（蛋白-蛋白、配体-配体、蛋白-配体）的成对距离
   - 对距离阈值内的原子对计算消息（MLP: [h_i || h_j || d_ij] -> msg）
   - 用 scatter_add 聚合消息到目标节点
   - 同时更新**节点特征**（残差连接 + LayerNorm）和**坐标**（标量权重 x 方向向量）
   - 关键区别：**同时更新蛋白和配体的坐标**，而非只更新配体

3. **坐标精修头**: 最终从节点特征预测额外的坐标位移，叠加到交互层的输出坐标上

In [ ]:
# ============================================================
# 模型架构 -- BidirectionalInteractionLayer
# ============================================================

class BidirectionalInteractionLayer(nn.Module):
    """
    双向交互层：蛋白-配体之间的消息传递。

    核心思想：
      1. 计算蛋白原子与配体原子之间的成对距离；
      2. 对距离阈值内的原子对计算消息；
      3. 同时更新蛋白和配体的节点特征；
      4. 同时更新蛋白和配体的坐标（通过学习位移向量）。

    这是 FlexPose 与其他对接模型的关键区别——
    其他模型（如 KarmaDock）只更新配体坐标，
    而 FlexPose 同时更新蛋白侧链和配体坐标。
    """
    def __init__(self, hidden_dim, cutoff):
        super().__init__()
        self.cutoff = cutoff

        # 消息 MLP: 输入 = [h_i || h_j || dist_ij]
        self.msg_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        # 节点更新 MLP: 输入 = [h_i || agg_msg]
        self.node_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        # 坐标位移预测: 从消息中预测标量权重，乘以方向向量得到位移
        self.coord_weight = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1),
        )

        self.layer_norm = nn.LayerNorm(hidden_dim)

    def forward(self, h_prot, coords_prot, h_lig, coords_lig):
        """
        参数:
          h_prot     : (N_p, hidden) 蛋白节点特征
          coords_prot: (N_p, 3)      蛋白原子坐标
          h_lig      : (N_l, hidden) 配体节点特征
          coords_lig : (N_l, 3)      配体原子坐标
        返回:
          h_prot_new, coords_prot_new, h_lig_new, coords_lig_new
        """
        N_p = h_prot.shape[0]

        # ---- 1. 拼接所有原子的特征和坐标 ----
        h_all = torch.cat([h_prot, h_lig], dim=0)          # (N_p+N_l, hidden)
        coords_all = torch.cat([coords_prot, coords_lig], dim=0)  # (N_p+N_l, 3)

        # ---- 2. 计算成对距离矩阵 ----
        diff = coords_all.unsqueeze(0) - coords_all.unsqueeze(1)  # (N, N, 3)
        dist = torch.norm(diff, dim=-1)                            # (N, N)

        # ---- 3. 筛选距离阈值内的交互对 (排除自身) ----
        mask = (dist < self.cutoff) & (dist > 1e-6)  # (N, N) bool

        # ---- 4. 计算消息并聚合 ----
        # 对每个原子 i，聚合来自邻居 j 的消息
        agg_msg = torch.zeros_like(h_all)           # (N, hidden)
        coord_update = torch.zeros_like(coords_all) # (N, 3)

        # 获取邻居对索引
        src_idx, dst_idx = torch.where(mask)  # src -> dst 的消息

        if src_idx.shape[0] > 0:
            h_src = h_all[src_idx]                    # (E, hidden)
            h_dst = h_all[dst_idx]                    # (E, hidden)
            d = dist[src_idx, dst_idx].unsqueeze(-1)  # (E, 1)
            direction = diff[dst_idx, src_idx]        # (E, 3) 从 dst 指向 src
            direction = direction / (d + 1e-8)        # 单位方向向量

            # 计算消息
            msg_input = torch.cat([h_src, h_dst, d], dim=-1)  # (E, hidden*2+1)
            msg = self.msg_mlp(msg_input)                     # (E, hidden)

            # 聚合消息到目标节点 (scatter_add)
            agg_msg = agg_msg.scatter_add(0, dst_idx.unsqueeze(-1).expand_as(msg), msg)

            # 计算坐标位移: 标量权重 * 方向向量
            w = self.coord_weight(msg)                  # (E, 1)
            weighted_dir = w * direction                # (E, 3)
            coord_update = coord_update.scatter_add(
                0, dst_idx.unsqueeze(-1).expand_as(weighted_dir), weighted_dir
            )

        # ---- 5. 更新节点特征 (残差连接 + LayerNorm) ----
        node_input = torch.cat([h_all, agg_msg], dim=-1)  # (N, hidden*2)
        h_all_new = h_all + self.node_mlp(node_input)     # 残差连接
        h_all_new = self.layer_norm(h_all_new)

        # ---- 6. 更新坐标 ----
        coords_all_new = coords_all + coord_update

        # ---- 7. 拆分蛋白和配体 ----
        h_prot_new = h_all_new[:N_p]
        h_lig_new = h_all_new[N_p:]
        coords_prot_new = coords_all_new[:N_p]
        coords_lig_new = coords_all_new[N_p:]

        return h_prot_new, coords_prot_new, h_lig_new, coords_lig_new

In [ ]:
# ============================================================
# 模型架构 -- ToyFlexPose 主模型
# ============================================================

class ToyFlexPose(nn.Module):
    """
    FlexPose 玩具模型 -- 联合坐标预测。

    核心思想：
      与 KarmaDock 等模型类似的消息传递架构，
      但同时更新蛋白质和配体的坐标，实现真正的柔性对接。

    架构:
      - prot_embed: Linear(10, 128)  蛋白原子嵌入
      - lig_embed:  Linear(10, 128)  配体原子嵌入
      - interaction_layers: 6 层双向交互层
      - prot_coord_head: Linear(128, 3)  蛋白原子位移预测
      - lig_coord_head:  Linear(128, 3)  配体原子位移预测

    训练流程：
      1. 输入加噪的蛋白和配体坐标；
      2. 经过 6 层交互层逐步修正坐标；
      3. 最终用坐标预测头输出额外的精细位移；
      4. 损失 = RMSD_lig + 0.5 * RMSD_prot (加权损失)。
    """
    def __init__(self, atom_dim=ATOM_FEAT_DIM, hidden_dim=HIDDEN_DIM,
                 n_layers=N_LAYERS, cutoff=DISTANCE_CUTOFF):
        super().__init__()
        # 1. 原子特征嵌入层 -- 蛋白和配体使用独立嵌入
        self.prot_embed = nn.Linear(atom_dim, hidden_dim)
        self.lig_embed = nn.Linear(atom_dim, hidden_dim)

        # 2. 双向交互层 -- 同时更新蛋白和配体的特征与坐标
        self.interaction_layers = nn.ModuleList([
            BidirectionalInteractionLayer(hidden_dim, cutoff)
            for _ in range(n_layers)
        ])

        # 3. 最终坐标精修头 -- 从特征预测额外的位移
        self.prot_coord_head = nn.Linear(hidden_dim, 3)
        self.lig_coord_head = nn.Linear(hidden_dim, 3)

    def forward(self, prot_feats, prot_coords_noised, lig_feats, lig_coords_noised):
        """
        参数:
          prot_feats        : (N_p, atom_dim) 蛋白原子特征
          prot_coords_noised: (N_p, 3)        蛋白加噪坐标 (输入)
          lig_feats         : (N_l, atom_dim) 配体原子特征
          lig_coords_noised : (N_l, 3)        配体加噪坐标 (输入)

        返回:
          prot_coords_pred  : (N_p, 3) 蛋白预测坐标
          lig_coords_pred   : (N_l, 3) 配体预测坐标
        """
        # Step 1: 嵌入原子特征
        h_prot = self.prot_embed(prot_feats)  # (N_p, hidden)
        h_lig = self.lig_embed(lig_feats)     # (N_l, hidden)

        # 初始坐标 = 加噪坐标
        coords_prot = prot_coords_noised
        coords_lig = lig_coords_noised

        # Step 2: 经过多层双向交互层，逐步修正特征和坐标
        for layer in self.interaction_layers:
            h_prot, coords_prot, h_lig, coords_lig = layer(
                h_prot, coords_prot, h_lig, coords_lig
            )

        # Step 3: 最终精修 -- 从特征预测额外的坐标位移
        prot_delta = self.prot_coord_head(h_prot)  # (N_p, 3)
        lig_delta = self.lig_coord_head(h_lig)     # (N_l, 3)

        prot_coords_pred = coords_prot + prot_delta
        lig_coords_pred = coords_lig + lig_delta

        return prot_coords_pred, lig_coords_pred


# ---- 实例化模型 ----
model = ToyFlexPose(
    atom_dim=ATOM_FEAT_DIM,
    hidden_dim=HIDDEN_DIM,
    n_layers=N_LAYERS,
    cutoff=DISTANCE_CUTOFF
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# 展示模型参数信息
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

model_info = pd.DataFrame({
    '指标': ['总参数量', '可训练参数量', '交互层数', '隐层维度', '设备'],
    '值': [f'{total_params:,}', f'{trainable_params:,}', N_LAYERS, HIDDEN_DIM, str(DEVICE)]
})
print("模型信息:")
display(model_info)
print(f"\n模型结构:\n{model}")

## 5. 训练

训练流程：
1. 输入加噪的蛋白和配体坐标
2. 模型前向传播，预测去噪后的坐标
3. 计算加权损失: `loss = RMSD_lig + 0.5 * RMSD_prot`（配体对接精度更重要）
4. 梯度裁剪（max_norm=10.0）防止训练不稳定
5. 每 20 轮在测试集上进行验证

In [ ]:
# ============================================================
# 训练循环
# ============================================================
print(f"开始训练 {N_EPOCHS} 轮...\n")

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    train_lig_rmsds = []
    train_prot_rmsds = []
    train_losses = []

    for prot_f, prot_c, prot_c_n, lig_f, lig_c, lig_c_n in train_loader:
        # 每个 batch 只有 1 个样本 (变长图)，去掉 batch 维度
        prot_f = prot_f.squeeze(0).to(DEVICE)       # (N_p, 10)
        prot_c = prot_c.squeeze(0).to(DEVICE)       # (N_p, 3) 真实坐标
        prot_c_n = prot_c_n.squeeze(0).to(DEVICE)   # (N_p, 3) 加噪坐标
        lig_f = lig_f.squeeze(0).to(DEVICE)         # (N_l, 10)
        lig_c = lig_c.squeeze(0).to(DEVICE)         # (N_l, 3) 真实坐标
        lig_c_n = lig_c_n.squeeze(0).to(DEVICE)     # (N_l, 3) 加噪坐标

        optimizer.zero_grad()

        # 前向传播: 从加噪坐标预测真实坐标
        prot_pred, lig_pred = model(prot_f, prot_c_n, lig_f, lig_c_n)

        # 损失 = RMSD_lig + 0.5 * RMSD_prot (配体更重要)
        rmsd_lig = compute_rmsd(lig_pred, lig_c)
        rmsd_prot = compute_rmsd(prot_pred, prot_c)
        loss = rmsd_lig + 0.5 * rmsd_prot

        loss.backward()
        # 梯度裁剪，防止训练不稳定
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()

        train_losses.append(loss.item())
        train_lig_rmsds.append(rmsd_lig.item())
        train_prot_rmsds.append(rmsd_prot.item())

    # ---- 每 20 轮验证一次 ----
    if epoch % 20 == 0 or epoch == 1:
        model.eval()
        val_lig_rmsds = []
        val_prot_rmsds = []
        with torch.no_grad():
            for prot_f, prot_c, prot_c_n, lig_f, lig_c, lig_c_n in test_loader:
                prot_f = prot_f.squeeze(0).to(DEVICE)
                prot_c = prot_c.squeeze(0).to(DEVICE)
                prot_c_n = prot_c_n.squeeze(0).to(DEVICE)
                lig_f = lig_f.squeeze(0).to(DEVICE)
                lig_c = lig_c.squeeze(0).to(DEVICE)
                lig_c_n = lig_c_n.squeeze(0).to(DEVICE)

                prot_pred, lig_pred = model(prot_f, prot_c_n, lig_f, lig_c_n)
                val_lig_rmsds.append(compute_rmsd(lig_pred, lig_c).item())
                val_prot_rmsds.append(compute_rmsd(prot_pred, prot_c).item())

        avg_train_loss = np.mean(train_losses) if train_losses else float('nan')
        avg_val_lig = np.mean(val_lig_rmsds) if val_lig_rmsds else float('nan')
        avg_val_prot = np.mean(val_prot_rmsds) if val_prot_rmsds else float('nan')
        print(f"  Epoch {epoch:>4d}/{N_EPOCHS}  |  "
              f"Train Loss: {avg_train_loss:.4f}  |  "
              f"Val Lig RMSD: {avg_val_lig:.4f}  |  "
              f"Val Prot RMSD: {avg_val_prot:.4f}")

## 6. 测试与可视化

在测试集上最终检验模型性能，同时计算噪声基线（不经过模型，直接用加噪坐标的 RMSD）作为对比：

- **配体 RMSD**: 预测配体坐标与真实坐标的偏差
- **蛋白 RMSD**: 预测蛋白侧链坐标与真实坐标的偏差
- **基线 RMSD**: 加噪坐标直接与真实坐标对比的偏差

关注指标：
- Mean/Median RMSD
- RMSD < 2A 的比例（高精度对接）
- RMSD < 5A 的比例（可接受的对接）

In [ ]:
# ============================================================
# 测试集检验
# ============================================================
print("在测试集上检验...\n")

model.eval()
test_lig_rmsds = []
test_prot_rmsds = []
# 同时记录噪声基线 (不经过模型直接用加噪坐标的 RMSD)
baseline_lig_rmsds = []
baseline_prot_rmsds = []

with torch.no_grad():
    for prot_f, prot_c, prot_c_n, lig_f, lig_c, lig_c_n in test_loader:
        prot_f = prot_f.squeeze(0).to(DEVICE)
        prot_c = prot_c.squeeze(0).to(DEVICE)
        prot_c_n = prot_c_n.squeeze(0).to(DEVICE)
        lig_f = lig_f.squeeze(0).to(DEVICE)
        lig_c = lig_c.squeeze(0).to(DEVICE)
        lig_c_n = lig_c_n.squeeze(0).to(DEVICE)

        prot_pred, lig_pred = model(prot_f, prot_c_n, lig_f, lig_c_n)

        test_lig_rmsds.append(compute_rmsd(lig_pred, lig_c).item())
        test_prot_rmsds.append(compute_rmsd(prot_pred, prot_c).item())
        baseline_lig_rmsds.append(compute_rmsd(lig_c_n, lig_c).item())
        baseline_prot_rmsds.append(compute_rmsd(prot_c_n, prot_c).item())

test_lig_rmsds = np.array(test_lig_rmsds)
test_prot_rmsds = np.array(test_prot_rmsds)
baseline_lig_rmsds = np.array(baseline_lig_rmsds)
baseline_prot_rmsds = np.array(baseline_prot_rmsds)

# ---- 计算统计指标 ----
mean_lig_rmsd = np.mean(test_lig_rmsds)
median_lig_rmsd = np.median(test_lig_rmsds)
pct_under_2 = np.mean(test_lig_rmsds < 2.0) * 100
pct_under_5 = np.mean(test_lig_rmsds < 5.0) * 100

# ---- 用 DataFrame 展示结果 ----
results_df = pd.DataFrame({
    '指标': ['样本数', 'Mean Ligand RMSD (A)', 'Median Ligand RMSD (A)',
            'RMSD < 2A (%)', 'RMSD < 5A (%)',
            '配体 RMSD 基线 (A)', '蛋白 RMSD 预测 (A)', '蛋白 RMSD 基线 (A)'],
    '值': [
        f'{len(test_lig_rmsds)}',
        f'{mean_lig_rmsd:.4f}',
        f'{median_lig_rmsd:.4f}',
        f'{pct_under_2:.1f}',
        f'{pct_under_5:.1f}',
        f'{baseline_lig_rmsds.mean():.4f} +/- {baseline_lig_rmsds.std():.4f}',
        f'{test_prot_rmsds.mean():.4f} +/- {test_prot_rmsds.std():.4f}',
        f'{baseline_prot_rmsds.mean():.4f} +/- {baseline_prot_rmsds.std():.4f}',
    ]
})
print("测试集结果:")
display(results_df)

In [ ]:
# ============================================================
# 可视化: 配体 RMSD + 蛋白 RMSD 双直方图
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 左图: 配体 RMSD 分布
ax = axes[0]
ax.hist(baseline_lig_rmsds, bins=20, alpha=0.5, label='Noised (baseline)', color='gray')
ax.hist(test_lig_rmsds, bins=20, alpha=0.7, label='Predicted', color='steelblue')
ax.axvline(test_lig_rmsds.mean(), color='steelblue', linestyle='--', linewidth=1.5,
           label=f'Pred mean={test_lig_rmsds.mean():.2f}')
ax.axvline(baseline_lig_rmsds.mean(), color='gray', linestyle='--', linewidth=1.5,
           label=f'Base mean={baseline_lig_rmsds.mean():.2f}')
ax.set_xlabel('Ligand RMSD (A)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Ligand RMSD Distribution', fontsize=13)
ax.legend(fontsize=9)

# 右图: 蛋白 RMSD 分布
ax = axes[1]
ax.hist(baseline_prot_rmsds, bins=20, alpha=0.5, label='Noised (baseline)', color='gray')
ax.hist(test_prot_rmsds, bins=20, alpha=0.7, label='Predicted', color='coral')
ax.axvline(test_prot_rmsds.mean(), color='coral', linestyle='--', linewidth=1.5,
           label=f'Pred mean={test_prot_rmsds.mean():.2f}')
ax.axvline(baseline_prot_rmsds.mean(), color='gray', linestyle='--', linewidth=1.5,
           label=f'Base mean={baseline_prot_rmsds.mean():.2f}')
ax.set_xlabel('Protein RMSD (A)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Protein Pocket RMSD Distribution', fontsize=13)
ax.legend(fontsize=9)

fig.suptitle('FlexPose Toy Model \u2014 Joint Coordinate Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 总结

本教程实现了 FlexPose 的玩具版本，展示了**柔性对接**的核心思想：

### 与半柔性对接的关键区别

| 特性 | 半柔性对接 (DiffDock, KarmaDock) | 柔性对接 (FlexPose) |
|------|-------------------------------|-------------------|
| 蛋白构象 | 固定不动 | 侧链可移动 |
| 预测目标 | 仅配体坐标 | 蛋白侧链 + 配体坐标 |
| 消息传递 | 单向 (配体从蛋白接收) | 双向 (蛋白-配体互相影响) |
| 坐标更新 | 仅更新配体 | 同时更新蛋白和配体 |

### 本教程的核心要点

1. **加噪-去噪训练范式**: 对蛋白侧链（小噪声 sigma=1.0A）和配体（大噪声 sigma=5.0A）施加不同强度的噪声，训练模型恢复真实坐标
2. **双向消息传递**: `BidirectionalInteractionLayer` 同时更新蛋白和配体的特征与坐标，实现真正的联合优化
3. **加权损失**: `loss = RMSD_lig + 0.5 * RMSD_prot`，配体对接精度权重更高
4. **坐标精修头**: 在交互层逐步修正的基础上，最终用线性层预测额外的精细位移

### 局限性

- 本玩具模型仅用 CASF-2016 核心集（285 个复合物）训练，数据量远不及原论文
- 原论文使用更复杂的坐标更新策略（如 IPA 注意力）和更大的模型
- 实际应用中需要考虑蛋白质骨架约束、侧链二面角采样等物理先验